<a href="https://colab.research.google.com/drive/1Xzio6Sbp42stOJvgErdsguvL54mHPibg?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![CuratorKIT](CuratorKIT_logo-TP.png)

---

# Synthetic generation

**Exporter:** `alpaca` &nbsp;|&nbsp; **Gates:** HallucinationGate + RewardGate (always active)

Converts a PDF into instruction-following training data. Choose your generation task below.

### Valid tasks for Alpaca export

| Task | What it does | Output |
|------|-------------|--------|
| `qa` | Generate grounded question-answer pairs from each chunk | 3 QA pairs per chunk |
| `evol` | Take existing instructions and evolve them through complexity strategies | Evolved instruction + answer |
| `cot` | Generate chain-of-thought reasoning with the answer | Reasoning steps + final answer |

All three produce `task_type="instruction_following"` → exported as `sft_alpaca.jsonl`.

### Pipeline
```
PDF → chunk → dedup → clean → [TASK] → HallucinationGate → RewardGate → DiversityGate → alpaca
```

In [3]:
# ═══════════════════════════════════════════════════════════════════════════
# 0. Setup — clone + install CuratorKIT (run once, then comment out)
# Prerequisite: SSH key added to your GitHub account with repo access.
# ═══════════════════════════════════════════════════════════════════════════
from pathlib import Path

repo_dir = Path.home() / "CuratorKIT"
if not repo_dir.exists():
    !git clone https://github.com/Lexsi-Labs/CuratorKIT.git {repo_dir}

!pip install -e "{repo_dir}[generation,embedding,hf,parquet]" -q

print(f"CuratorKIT installed from {repo_dir}")

# !pip install "/content/curatorkit.tar.gz[generation,embedding,hf,parquet,pdf-gpu]" nest-asyncio -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 17.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.6/786.6 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

## 1. Configuration — pick your task

In [4]:
import urllib.request
urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762", "attention_is_all_you_need.pdf")
print("Downloaded.")

Downloaded.


## LLM Backend Setup

Configure your LLM endpoint before running. CuratorKIT uses [LiteLLM](https://docs.litellm.ai) under the hood, so any OpenAI-compatible endpoint works.

### Option A — vLLM (recommended for speed)

```bash
# Serve a local model with vLLM
pip install vllm
vllm serve Qwen/Qwen2.5-7B-Instruct --port 8000
```

Then set in the config cell below:
```python
MODEL    = "openai/Qwen/Qwen2.5-7B-Instruct"
API_BASE = "http://localhost:8000/v1"
```

### Option B — Ollama (easiest local setup)

```bash
# Install Ollama: https://ollama.com
ollama pull qwen2.5:7b
ollama serve  # starts on port 11434 by default
```

Then set:
```python
MODEL    = "ollama/qwen2.5:7b"
API_BASE = ""  # Ollama backend is auto-detected from the model prefix
```

### Option C — Any OpenAI-compatible API

```python
MODEL    = "openai/your-model-name"
API_BASE = "https://your-api-endpoint/v1"
# Set API key via env var: export OPENAI_API_KEY=your-key
# Or pass directly: llm_api_key="your-key" in CuratorConfig
```


In [5]:
MODEL    = "openai/Qwen/Qwen3-8B"
JUDGE    = "openai/Qwen/Qwen3-8B"
API_BASE = " "

In [6]:
from pathlib import Path
from curatorkit import Curator, CuratorConfig

# ═══════════════════════════════════════════════════════════════════════════
# Pick your task: "qa" | "evol" | "cot"
# ═══════════════════════════════════════════════════════════════════════════
TASK = "qa"

# ── PDF source ────────────────────────────────────────────────────────────
PDF_PATH = "attention_is_all_you_need.pdf"  # ← or replace with your own PDF


# ── Pipeline params ───────────────────────────────────────────────────────
MAX_CHUNKS  = 30
CONCURRENCY = 32
OUTPUT_DIR  = Path(f"output/synthetic_generation_{TASK}")

print(f"Task   : {TASK}")
print(f"PDF    : {PDF_PATH}")
print(f"Model  : {MODEL}")
print(f"Output : {OUTPUT_DIR}/")

Task   : qa
PDF    : attention_is_all_you_need.pdf
Model  : openai/Qwen/Qwen3-8B
Output : output/synthetic_generation_qa/


## 2. Build config per task

In [7]:
api_base = API_BASE if API_BASE else None

# ── Shared config across all alpaca tasks ─────────────────────────────────
shared = dict(
    dataset={"name": PDF_PATH, "max_samples": MAX_CHUNKS},
    pdf_chunk_strategy="heading",
    pdf_chunk_max_tokens=512,
    pdf_chunk_overlap_tokens=50,
    dedup="minhash",
    minhash_threshold=0.85,
    clean=True,
    llm_model=MODEL,
    llm_api_base=api_base,
    llm_max_tokens=2048,
    llm_concurrency=CONCURRENCY,
    generation_concurrency=CONCURRENCY,
    llm_extra_body={"chat_template_kwargs": {"enable_thinking": True}},
    judge_llm_model=JUDGE,
    judge_llm_api_base=api_base,
    judge_llm_temperature=0.1,
    judge_concurrency=CONCURRENCY,
    judge_llm_extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    hallucination_threshold=0.7,
    reward_threshold=0.75,
    reward_dimensions=["helpfulness", "honesty", "instruction_following"],
    diversity_threshold=0.92,
    embedding_device="cuda",
    embedding_batch_size=256,
    export_formats=["alpaca"],
    output_dir=OUTPUT_DIR,
)

# ── Task-specific config ──────────────────────────────────────────────────
if TASK == "qa":
    task_config = dict(
        generation_task="qa",
        num_questions=3,
        difficulty="medium",# ═══════════════════════════════════════════════════════════════════════════
# 0. Setup — clone + install CuratorKIT (run once, then comment out)
# Prerequisite: SSH key added to your GitHub account with repo access.
# ═══════════════════════════════════════════════════════════════════════════
from pathlib import Path

repo_dir = Path.home() / "CuratorKIT"
if not repo_dir.exists():
    !git clone https://github.com/Lexsi-Labs/CuratorKIT.git {repo_dir}

!pip install -e "{repo_dir}[generation,embedding,hf,parquet]" -q

print(f"CuratorKIT installed from {repo_dir}")
    )

elif TASK == "evol":
    task_config = dict(
        generation_task="evol",
        num_evolutions=2,
        generate_answers=True,
    )

elif TASK == "cot":
    task_config = dict(
        generation_task="cot",
        cot_mode="generate",
    )

else:
    raise ValueError(f"Unknown TASK: {TASK}. Choose: qa | evol | cot")

config = CuratorConfig(**shared, **task_config, llm_api_key="")
print(f"Generation task: {config.generation_task}")
print(f"Export format  : {config.export_formats}")
print("\nRunning...")
result = Curator(config).run()
result.print_summary()

Generation task: qa
Export format  : ['alpaca']

Running...


2026-06-12 05:48:37.403 | INFO     | mineru.backend.pipeline.pipeline_analyze:doc_analyze_streaming:209 - Pipeline processing-window multi-file run. doc_count=1, total_pages=15, window_size=64, total_batches=1
2026-06-12 05:48:37.404 | DEBUG    | mineru.utils.pdf_image_tools:_load_images_from_pdf_bytes_range:289 - PDF image rendering uses 1 processes for pages 1-15: [(0, 14)]
2026-06-12 05:48:37.405 | DEBUG    | mineru.utils.pdf_image_tools:_create_pdf_render_executor:161 - PDF image rendering switches multiprocessing start method from fork to spawn
2026-06-12 05:48:37.407 | DEBUG    | mineru.utils.pdf_image_tools:_get_pdf_render_executor:218 - Created persistent PDF render executor with max_workers=2
2026-06-12 05:48:40.034 | INFO     | mineru.backend.pipeline.pipeline_analyze:doc_analyze_streaming:264 - Pipeline processing window batch 1/1: 15/15 pages, batch_pages=15, doc_slices=doc0:1-15
2026-06-12 05:48:40.037 | INFO     | mineru.backend.pipeline.pipeline_analyze:batch_image_analy

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

DiversityGate: 100%|██████████| 6/6 [00:00<00:00, 11597.15sample/s]

────────────────────────────────────────────
  passed   :        6
  rejected :      110
  time     :   185.3s
  output   : output/synthetic_generation_qa
────────────────────────────────────────────


## 3. Inspect

In [8]:
from collections import Counter

if result.passed:
    print(f"Passed: {len(result.passed):,}  |  Rejected: {len(result.rejected):,}\n")
    for i, s in enumerate(result.passed[:3]):
        print(f"── Sample {i+1} ({s.task_type}) ──")
        print(f"  Q: {s.instruction[:120]}")
        print(f"  A: {s.output[:120]}")
        for rec in s.provenance_chain:
            if rec.step_name == "HallucinationGate":
                print(f"  grounding: {rec.notes.get('grounding_score', '-')}  ({rec.notes.get('verdict', '-')})")
            if rec.step_name == "RewardGate":
                print(f"  reward:    {rec.notes.get('reward_score', '-')}")
        print()

if result.rejected:
    reasons = Counter(r.rejection_reason for r in result.rejected)
    print("Rejection reasons:")
    for reason, count in reasons.most_common():
        print(f"  {count:>4}  {reason}")

Passed: 6  |  Rejected: 110

── Sample 1 (instruction_following) ──
  Q: u
  A: lity question-answer pairs from source text.
   - **Task:** Generate exactly 3 question-answer pairs from the provided t
  grounding: 1.0  (grounded)
  reward:    1.0

── Sample 2 (instruction_following) ──
  Q: (Conceptu
  A: l):* How do attention mechanisms typically function in sequence modeling tasks, and how does the proposed Transformer ar
  grounding: 1.0  (grounded)
  reward:    0.95

── Sample 3 (instruction_following) ──
  Q: (Conceptu
  A: l):* How does the Transformer's approach to computing dependencies between input and output positions differ from convol
  grounding: 1.0  (grounded)
  reward:    0.95

Rejection reasons:
    42  hallucination_contract_failed:0.00
    38  below_reward_threshold:0.00
    23  below_reward_threshold:0.10
     4  table_stubbed_phase_0
     2  below_reward_threshold:0.05
     1  below_reward_threshold:0.35


## 4. Task comparison

| Task | Gates check | Common failure | Best for |
|------|------------|----------------|----------|
| `qa` | Grounding + quality of both Q and A | GENERATOR_PARAMETRIC (answer ignores source) | General SFT from documents |
| `evol` | Grounding + quality of evolved instruction AND answer | INSTRUCTION_QUALITY (evolved instruction becomes ambiguous) | Increasing instruction complexity |
| `cot` | Grounding of answer section + overall quality | GENERATOR_PARAMETRIC (reasoning pulls in outside facts) | Training reasoning capabilities |

### Also valid for alpaca export (not shown)
- **ShareGPT format:** change `export_formats=["sharegpt"]` for conversational format
- **Multi-turn:** use `generation_task="multiturn"` then `export_formats=["sharegpt"]`